In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

MODULE 0

In [ ]:
!pip install peft -q

In [1]:
# Module 0 (V4 PEFT): Setup, Imports, PEFT Install, GPU Check, V4 Config

# --- Install PEFT library ---
print("Installing PEFT library...")
# Optional: Install bitsandbytes if considering QLoRA later
# !pip install bitsandbytes -q
print("PEFT potentially installed.")

import json
import os
import gzip
import pandas as pd
import torch # PyTorch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM, # PyTorch model
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
# Import PEFT classes
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import pprint
import time
import re # For parsing in Module 3.5

print("\n--- Module 0 (V4 PEFT): Setup & Imports ---")
print(f"PyTorch version: {torch.__version__}")

# --- Check for GPU ---
print("\nChecking for GPU availability for PyTorch...")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU Available! Device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("ERROR: No GPU detected by PyTorch. Cannot train effectively.")
    raise SystemExit("GPU not available.")

# --- Configuration (V4 PEFT - t5-base, 2 files, Filtered) ---
dataset_base_path = '/kaggle/input/wikipedia-structured-contents/'
# Input files for Module 3
input_file_path_0 = os.path.join(dataset_base_path, 'enwiki_namespace_0/enwiki_namespace_0_0.jsonl')
input_file_path_1 = os.path.join(dataset_base_path, 'enwiki_namespace_0/enwiki_namespace_0_1.jsonl')
IS_COMPRESSED = False
NUM_ARTICLES_TO_PROCESS = None # Process all articles in the files

# Intermediate V3 Combined file path (Output of M3, Input to M3.5)
output_csv_path_v3 = '/kaggle/working/wiki_structure_data_v3_combined.csv'

# Output file paths for V4 (Filtered) data (Output of M3.5, Input to M4)
output_csv_path_v4 = '/kaggle/working/wiki_structure_data_v4_filtered.csv'
train_csv_path_v4 = '/kaggle/working/train_data_v4.csv'
val_csv_path_v4 = '/kaggle/working/val_data_v4.csv'

# Model checkpoint and prefix
model_checkpoint = "t5-base" # Using t5-base
task_prefix = "generate sections: "

# PEFT / LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q", "v"] # Target query/value matrices in T5 attention

# Training Hyperparameters (V4 PEFT)
EPOCHS = 3 # Using 3 epochs for PEFT on filtered data
PER_DEVICE_BATCH_SIZE = 16 # PEFT should allow BS=16 for t5-base
GRADIENT_ACCUMULATION_STEPS = 1
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 2 # Keep best + latest checkpoints
FP16_ENABLED = torch.cuda.is_available() # Use Mixed Precision

# Output directory for V4 PEFT adapter
peft_output_dir_v4 = f"/kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-{model_checkpoint}-PEFT"
logging_dir_pt_v4 = f"{peft_output_dir_v4}/logs"
final_adapter_dir_v4 = f"{peft_output_dir_v4}/final_adapter" # Where final adapter will be saved


print(f"\n--- Configuration for V4 PEFT Run ---")
print(f"Input files (M3): \n  - {input_file_path_0}\n  - {input_file_path_1}")
print(f"Intermediate Combined Output (M3): {output_csv_path_v3}")
print(f"Filtered Data Output (M3.5): {train_csv_path_v4}, {val_csv_path_v4}")
print(f"Using input feature: 'name [SEP] description'")
print(f"Using target structure: Hierarchical list -> Formatted String '(level: title) | ...'")
print(f"Base model: {model_checkpoint}")
print(f"PEFT Method: LoRA (r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES})")
print(f"Epochs (M4): {EPOCHS}")
print(f"Batch Size (Per Device M4): {PER_DEVICE_BATCH_SIZE}")
print(f"Effective Batch Size (M4): {EFFECTIVE_BATCH_SIZE}")
print(f"Learning Rate (M4): {LEARNING_RATE}")
print(f"FP16 Enabled (M4): {FP16_ENABLED}")
print(f"Adapter Output/Checkpoint Dir (M4): {peft_output_dir_v4}")
print(f"Final Adapter Save Dir (M4): {final_adapter_dir_v4}")
print("---------------------------------")

# --- Define Helper Functions (Modules 1 & 2) ---

# Module 1 Helper
def parse_article_line(line):
    try:
        if not isinstance(line, str): line = line.decode('utf-8')
        article_data = json.loads(line.strip())
        return article_data
    except: return None

# Module 2 (Revised) Helper: Recursive Traversal
def _traverse_parts(parts_list, current_level, results_list):
    if not isinstance(parts_list, list): return
    for part in parts_list:
        if isinstance(part, dict):
            part_type = part.get('type')
            if part_type == 'section':
                title = part.get('title', part.get('name', ''))
                if title: results_list.append((current_level, title))
                nested_parts = part.get('has_parts', [])
                if nested_parts: _traverse_parts(nested_parts, current_level + 1, results_list)

# Module 2 (Revised) Main Function: Hierarchical Extraction
def extract_hierarchical_structure(article_data):
    structure = []
    sections = article_data.get('sections', [])
    if not isinstance(sections, list) or not sections: return structure
    start_index = 0
    if isinstance(sections[0], dict) and sections[0].get('name') == 'Abstract': start_index = 1
    for i in range(start_index, len(sections)):
        section = sections[i]
        if isinstance(section, dict):
            title = section.get('name')
            if title:
                structure.append((1, title))
                parts = section.get('has_parts', [])
                if parts: _traverse_parts(parts, current_level=2, results_list=structure)
    return structure

# Target Formatting Function (needed for Module 3)
def format_hierarchical_target(structure_list):
    if not structure_list: return ""
    return " | ".join([f"({level}: {title})" for level, title in structure_list])


print("\n--- Modules 1 & 2 Functions Defined ---")
print("Ready for Module 3 (Data Preparation V3 - Rerun Needed)")

Installing PEFT library...
PEFT potentially installed.


2025-04-20 07:25:52.487441: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745133952.510474     246 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745133952.517432     246 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered



--- Module 0 (V4 PEFT): Setup & Imports ---
PyTorch version: 2.5.1+cu124

Checking for GPU availability for PyTorch...
GPU Available! Device: Tesla T4

--- Configuration for V4 PEFT Run ---
Input files (M3): 
  - /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_0.jsonl
  - /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_1.jsonl
Intermediate Combined Output (M3): /kaggle/working/wiki_structure_data_v3_combined.csv
Filtered Data Output (M3.5): /kaggle/working/train_data_v4.csv, /kaggle/working/val_data_v4.csv
Using input feature: 'name [SEP] description'
Using target structure: Hierarchical list -> Formatted String '(level: title) | ...'
Base model: t5-base
PEFT Method: LoRA (r=16, alpha=32, modules=['q', 'v'])
Epochs (M4): 3
Batch Size (Per Device M4): 16
Effective Batch Size (M4): 16
Learning Rate (M4): 0.0001
FP16 Enabled (M4): True
Adapter Output/Checkpoint Dir (M4): /kaggle/working/t5-wiki-structure-generator-pt-v4-

In [2]:
# Module 3 (Revised V3): Data Preparation (Rerun) - Create Combined V3 Data

import json
import os
import gzip
import pandas as pd
from tqdm.notebook import tqdm
import time

# Ensure functions parse_article_line, extract_hierarchical_structure,
# format_hierarchical_target are available from Module 0 cell.

# Also ensure configuration variables input_file_path_0, input_file_path_1, IS_COMPRESSED,
# NUM_ARTICLES_TO_PROCESS, output_csv_path_v3 are available from Module 0 cell.

print("--- Starting Module 3 (Revised V3): Data Preparation (Rerun) ---")
print(f"Processing File 1: {input_file_path_0}")
print(f"Processing File 2: {input_file_path_1}")
print(f"Outputting combined data to: {output_csv_path_v3}")

# --- Helper Function to Process One File ---
# (Ensure this function is defined - it was included in the Module 0 V4 PEFT Setup)
def process_single_file(filepath, num_articles=None):
    """Reads one input file and returns processed data as a list of dicts."""
    processed_data_list = []
    processed_count = 0
    skipped_count = 0
    open_func = gzip.open if IS_COMPRESSED else open
    print(f"\nStarting processing for: {filepath}")
    try:
        with open_func(filepath, 'rt', encoding='utf-8') as f:
            iterator = tqdm(f, desc=f"Processing {os.path.basename(filepath)}")
            for line in iterator:
                article = parse_article_line(line)
                if article:
                    article_name = article.get('name')
                    description = article.get('description', '')
                    if not article_name: skipped_count += 1; continue
                    if not isinstance(description, str): description = ''

                    input_text = f"{article_name} [SEP] {description}".strip()
                    hierarchical_structure = extract_hierarchical_structure(article) # Defined in Module 0

                    if not hierarchical_structure: skipped_count += 1; continue

                    target_structure_str = format_hierarchical_target(hierarchical_structure) # Defined in Module 0

                    processed_data_list.append({
                        'input_text': input_text,
                        'target_structure': target_structure_str
                    })
                    processed_count += 1
                else: skipped_count += 1
                if num_articles is not None and processed_count >= num_articles: break
            print(f"Finished {filepath}. Extracted: {processed_count}, Skipped: {skipped_count}")
            return processed_data_list
    except FileNotFoundError: print(f"Error: File not found at {filepath}"); return None
    except Exception as e: print(f"An error occurred during processing {filepath}: {e}"); import traceback; traceback.print_exc(); return None

# --- Main Processing ---
start_module_time = time.time()
data_part0 = process_single_file(input_file_path_0, NUM_ARTICLES_TO_PROCESS)
data_part1 = process_single_file(input_file_path_1, NUM_ARTICLES_TO_PROCESS)

combined_data = []
if data_part0: combined_data.extend(data_part0)
if data_part1: combined_data.extend(data_part1)

print(f"\nTotal combined examples: {len(combined_data)}")

# --- Save Combined Data ---
if combined_data:
    print("\nConverting combined data to DataFrame...")
    df_v3 = pd.DataFrame(combined_data)
    print(f"Combined DataFrame shape: {df_v3.shape}")
    print(f"\nSaving combined (V3) data to {output_csv_path_v3}...")
    try:
        df_v3.to_csv(output_csv_path_v3, index=False)
        print("Saved.")
    except Exception as e:
        print(f"ERROR saving combined CSV: {e}")
else:
    print("\nNo data was processed. Cannot save combined file.")

end_module_time = time.time()
print(f"\nTotal Module 3 V3 time: {end_module_time - start_module_time:.2f} seconds.")
print("\n--- Module 3 (Revised V3): Data Preparation Complete ---")
print("Ready for Module 3.5 (Filtering)")

--- Starting Module 3 (Revised V3): Data Preparation (Rerun) ---
Processing File 1: /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_0.jsonl
Processing File 2: /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_1.jsonl
Outputting combined data to: /kaggle/working/wiki_structure_data_v3_combined.csv

Starting processing for: /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_0.jsonl


Processing enwiki_namespace_0_0.jsonl: 0it [00:00, ?it/s]

Finished /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_0.jsonl. Extracted: 187063, Skipped: 122446

Starting processing for: /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_1.jsonl


Processing enwiki_namespace_0_1.jsonl: 0it [00:00, ?it/s]

Finished /kaggle/input/wikipedia-structured-contents/enwiki_namespace_0/enwiki_namespace_0_1.jsonl. Extracted: 187080, Skipped: 124518

Total combined examples: 374143

Converting combined data to DataFrame...
Combined DataFrame shape: (374143, 2)

Saving combined (V3) data to /kaggle/working/wiki_structure_data_v3_combined.csv...
Saved.

Total Module 3 V3 time: 50.70 seconds.

--- Module 3 (Revised V3): Data Preparation Complete ---
Ready for Module 3.5 (Filtering)


In [3]:
# Module 3.5: Data Filtering

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import time
import re # Using regex for slightly more robust parsing
import os # For file check

# Ensure config vars output_csv_path_v3, output_csv_path_v4,
# train_csv_path_v4, val_csv_path_v4 are available from Module 0 cell.

print("--- Starting Module 3.5: Data Filtering ---")

# --- Filtering Criteria (Review the output stats and ADJUST if needed) ---
MIN_SECTIONS = 3 # Keep articles with at least 3 sections total
MIN_MAX_DEPTH = 1 # Keep articles with max depth >= 1 (i.e., allow flat structures)
# Consider changing MIN_MAX_DEPTH = 2 if you want to force inclusion of subsections

print(f"Filtering Criteria: Min Sections >= {MIN_SECTIONS}, Min Max Depth >= {MIN_MAX_DEPTH}")
print(f"Loading combined data from: {output_csv_path_v3}")

# --- Load Data ---
if not os.path.exists(output_csv_path_v3):
    print(f"ERROR: Input file not found: {output_csv_path_v3}")
    print("Please run Module 3 first to generate the combined data.")
    raise SystemExit("Input data for filtering not found.")

try:
    df_v3 = pd.read_csv(output_csv_path_v3).astype(str)
    # Ensure target_structure is treated as string, handle potential loading issues
    df_v3['target_structure'] = df_v3['target_structure'].fillna('')
    df_v3.dropna(subset=['input_text'], inplace=True) # Only drop if input is missing
    print(f"\nLoaded V3 combined data. Shape: {df_v3.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    raise SystemExit("Data loading error.")


# --- Analysis Function ---
def analyze_structure(structure_str):
    """Parses the target string and returns structure metrics."""
    num_sections = 0
    max_depth = 0
    # Ensure input is a string
    if not isinstance(structure_str, str) or pd.isna(structure_str) or not structure_str:
        return 0, 0

    # Regex to find "(level: title)" parts - improved slightly
    pattern = re.compile(r'\((?P<level>\d+):\s*(?P<title>.*?)\)(?=\s*\||\s*$)')
    levels_found = []
    try:
        for match in pattern.finditer(structure_str):
            try:
                level = int(match.group('level'))
                levels_found.append(level)
            except (ValueError, IndexError):
                # print(f"Warning: Could not parse level from match: {match.group(0)}")
                pass # Ignore malformed parts
    except Exception as e:
         # print(f"Regex error on string: {structure_str[:100]}... Error: {e}")
         return 0,0 # Return default on regex error

    if levels_found:
        num_sections = len(levels_found)
        max_depth = max(levels_found) if levels_found else 0

    return num_sections, max_depth

# --- Apply Analysis ---
print("\nAnalyzing target structures (this might take a minute)...")
start_analysis_time = time.time()

# Use list comprehension and DataFrame constructor for potential speedup vs apply
analysis_results = [analyze_structure(s) for s in df_v3['target_structure']]
analysis_df = pd.DataFrame(analysis_results, index=df_v3.index, columns=['num_sections', 'max_depth'])
df_v3 = pd.concat([df_v3, analysis_df], axis=1)

end_analysis_time = time.time()
print(f"Analysis complete. Time: {end_analysis_time - start_analysis_time:.2f} seconds.")

# --- Show Distributions (FOR USER REVIEW) ---
print("\n--- Review Required: Distribution of Structure Metrics ---")
print("Number of Sections (`num_sections`):")
print(df_v3['num_sections'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))
print("\nMaximum Depth (`max_depth`):")
print(df_v3['max_depth'].describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))
print("\nMax Depth Value Counts (%):")
# Calculate percentage, sort by index (depth level)
depth_counts = df_v3['max_depth'].value_counts(normalize=True).sort_index()
print(depth_counts[depth_counts > 0.001]) # Show levels with > 0.1% frequency
if (depth_counts <= 0.001).any(): print("... (other depths < 0.1%)")
print("-----------------------------------------------------------")
# --- Add this block AFTER the .describe() printouts in Module 3.5 ---
print("\n--- Checking Specific Filter Counts ---")
        
        # Check how many meet the MIN_SECTIONS criteria
count_min_sections = 0
if 'num_sections' in df_v3.columns:
    count_min_sections = (df_v3['num_sections'] >= MIN_SECTIONS).sum()
        
        # Optionally, check how many meet BOTH criteria
    count_both_criteria = 0
    if 'num_sections' in df_v3.columns and 'max_depth' in df_v3.columns:
        count_both_criteria = ((df_v3['num_sections'] >= MIN_SECTIONS) & (df_v3['max_depth'] >= MIN_MAX_DEPTH)).sum()

        total_articles = len(df_v3)
        if total_articles > 0:
            percent_min_sections = (count_min_sections / total_articles) * 100
            percent_both_criteria = (count_both_criteria / total_articles) * 100
            
            print(f"Articles with >= {MIN_SECTIONS} sections: {count_min_sections} / {total_articles} ({percent_min_sections:.2f}%)")
            print(f"Articles with >= {MIN_SECTIONS} sections AND >= {MIN_MAX_DEPTH} max depth: {count_both_criteria} / {total_articles} ({percent_both_criteria:.2f}%)")
        else:
            print("No articles loaded to check criteria.")
            
        print("-------------------------------------")
        # The filtering code below this will use these criteria.
        # Reminder: Adjust MIN_SECTIONS / MIN_MAX_DEPTH at the top of the cell and re-run if needed.
print(f"Current Filter Settings: MIN_SECTIONS = {MIN_SECTIONS}, MIN_MAX_DEPTH = {MIN_MAX_DEPTH}")
print("--> Review the distributions above. If you want to change the filter thresholds,")
print("--> modify the MIN_SECTIONS / MIN_MAX_DEPTH variables near the TOP of THIS cell and RE-RUN the cell.")
print("--> Proceeding with current filter settings...")
time.sleep(5) # Add a small pause for user to read before filtering

# --- Apply Filtering ---
print(f"\nApplying filter: num_sections >= {MIN_SECTIONS} AND max_depth >= {MIN_MAX_DEPTH}")
original_count = len(df_v3)

# Ensure columns exist before filtering
if 'num_sections' in df_v3.columns and 'max_depth' in df_v3.columns:
    df_v4_filtered = df_v3[
        (df_v3['num_sections'] >= MIN_SECTIONS) &
        (df_v3['max_depth'] >= MIN_MAX_DEPTH)
    ].copy()
    filtered_count = len(df_v4_filtered)

    if original_count > 0:
        percent_kept = (filtered_count / original_count) * 100
        print(f"Filtering complete. Original: {original_count}, Filtered (V4): {filtered_count} ({percent_kept:.2f}% kept)")
    else:
        print("Filtering complete. Original dataset was empty.")

    # Drop analysis columns
    df_v4_filtered = df_v4_filtered[['input_text', 'target_structure']]
else:
    print("ERROR: Analysis columns ('num_sections', 'max_depth') not found. Cannot filter.")
    df_v4_filtered = pd.DataFrame() # Create empty dataframe

# --- Save and Split Filtered Data ---
if not df_v4_filtered.empty:
    print(f"\nSaving filtered (V4) data to {output_csv_path_v4}...")
    df_v4_filtered.to_csv(output_csv_path_v4, index=False)
    print("Saved.")
    print("\nSplitting filtered (V4) data...")
    if len(df_v4_filtered) > 1:
         train_df_v4, val_df_v4 = train_test_split(df_v4_filtered, test_size=0.2, random_state=42)
         print(f"Train set shape (V4): {train_df_v4.shape}")
         print(f"Validation set shape (V4): {val_df_v4.shape}")
         print(f"\nSaving train data (V4) to {train_csv_path_v4}...")
         train_df_v4.to_csv(train_csv_path_v4, index=False)
         print("Saved.")
         print(f"\nSaving validation data (V4) to {val_csv_path_v4}...")
         val_df_v4.to_csv(val_csv_path_v4, index=False)
         print("Saved.")
    else: print("Not enough filtered data to split.")
    print("\n--- Module 3.5: Data Filtering Complete ---")
    print("Ready for Module 4 (PEFT Training on V4 Data)")
else:
    print("\nFiltered DataFrame is empty. No data saved. Check filtering criteria or input data.")

--- Starting Module 3.5: Data Filtering ---
Filtering Criteria: Min Sections >= 3, Min Max Depth >= 1
Loading combined data from: /kaggle/working/wiki_structure_data_v3_combined.csv

Loaded V3 combined data. Shape: (374143, 2)

Analyzing target structures (this might take a minute)...
Analysis complete. Time: 1.66 seconds.

--- Review Required: Distribution of Structure Metrics ---
Number of Sections (`num_sections`):
count    374143.000000
mean          3.739645
std           4.389038
min           1.000000
10%           1.000000
25%           1.000000
50%           3.000000
75%           4.000000
90%           7.000000
95%          11.000000
99%          20.000000
max         236.000000
Name: num_sections, dtype: float64

Maximum Depth (`max_depth`):
count    374143.000000
mean          1.330045
std           0.557143
min           1.000000
10%           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
90%           2.000000
95%           2.000000
99%    

In [ ]:
!pip install bitsandbytes -q

In [ ]:
!pip install accelerate -q

In [4]:
# Upgrade bitsandbytes
print("Upgrading bitsandbytes library...")
!pip install --upgrade bitsandbytes -q
import bitsandbytes
print(f"bitsandbytes upgraded. Version: {bitsandbytes.__version__}")

Upgrading bitsandbytes library...
bitsandbytes upgraded. Version: 0.45.5


In [5]:
# Module 4 (Revised V4 - QLoRA): Training t5-base with QLoRA

import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM, # PyTorch model
    BitsAndBytesConfig, # For quantization
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
# Import PEFT classes
from peft import LoraConfig, get_peft_model, TaskType

import numpy as np
import time
import os

print("--- Setting up Module 4 (V4 QLoRA): Training t5-base with 4-bit Quantization + LoRA ---")
print(f"PyTorch version: {torch.__version__}")

# --- Check for GPU ---
print("\nChecking for GPU availability for PyTorch...")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU Available! Device: {torch.cuda.get_device_name(0)}")
    # T4 GPU check
    gpu_name = torch.cuda.get_device_name(0)
    if "T4" in gpu_name:
        print("Tesla T4 detected, proceeding with QLoRA setup.")
    else:
        print(f"Warning: GPU is {gpu_name}, not T4. Performance/compatibility may vary.")
else:
    device = torch.device("cpu")
    print("ERROR: No GPU detected by PyTorch. QLoRA requires GPU.")
    raise SystemExit("GPU not available.")

# --- Configuration (V4 QLoRA - t5-base) ---
# Paths from previous steps (ensure V4 data exists)
train_csv_path = '/kaggle/working/train_data_v4.csv'
val_csv_path = '/kaggle/working/val_data_v4.csv'

model_checkpoint = "t5-base" # Using t5-base
task_prefix = "generate sections: "

# PEFT / LoRA Configuration (same as before)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q", "v"]

# --- Quantization Configuration ---
# Setup 4-bit quantization using bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Use NF4 (Normal Float 4) quantization
    bnb_4bit_compute_dtype=torch.float16, # Compute type for T4 compatibility (FP16)
    bnb_4bit_use_double_quant=True, # Optional: nested quantization for more savings
)
print("\nBitsAndBytesConfig for 4-bit quantization defined.")

# Training Hyperparameters (V4 QLoRA)
EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 16 # Keep batch size 16 - should fit with QLoRA
GRADIENT_ACCUMULATION_STEPS = 1
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 1e-4 # Keep the PEFT learning rate
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 2
FP16_ENABLED = True # Use FP16 with T4

# Output Directories for QLoRA run
qlora_output_dir_v4 = f"/kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-{model_checkpoint}-QLoRA"
logging_dir_pt_v4 = f"{qlora_output_dir_v4}/logs"
final_adapter_dir_v4 = f"{qlora_output_dir_v4}/final_adapter"

print(f"\n--- Configuration for V4 QLoRA Run (t5-base) ---")
print(f"Loading V4 data from {train_csv_path} and {val_csv_path}")
print(f"Using base model: {model_checkpoint}")
print(f"Quantization: 4-bit NF4, Compute dtype: float16")
print(f"PEFT Method: LoRA (r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES})")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size (Per Device): {PER_DEVICE_BATCH_SIZE}")
print(f"Effective Batch Size: {EFFECTIVE_BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"FP16 Enabled: {FP16_ENABLED}")
print(f"Adapter Output/Checkpoint Dir: {qlora_output_dir_v4}")
print(f"Final Adapter Save Dir: {final_adapter_dir_v4}")
print("---------------------------------")


# --- Load Data ---
# (Make sure V4 CSVs exist from Module 3.5)
try:
    train_df_v4 = pd.read_csv(train_csv_path_v4).astype(str)
    val_df_v4 = pd.read_csv(val_csv_path_v4).astype(str)
    train_df_v4.dropna(subset=['input_text', 'target_structure'], inplace=True)
    val_df_v4.dropna(subset=['input_text', 'target_structure'], inplace=True)
    print(f"\nLoaded V4 train data: {train_df_v4.shape}") # ~154k
    print(f"Loaded V4 validation data: {val_df_v4.shape}") # ~39k
    raw_datasets_v4 = DatasetDict({
        'train': Dataset.from_pandas(train_df_v4),
        'validation': Dataset.from_pandas(val_df_v4)
    })
    print("\nConverted V4 data to Hugging Face DatasetDict.")
except FileNotFoundError as e:
    print(f"Error loading V4 data CSV files: {e}"); raise SystemExit("V4 Data loading failed.")
except Exception as e:
     print(f"An error occurred loading V4 data: {e}"); raise SystemExit("V4 Data loading failed.")

# --- Load Tokenizer ---
print(f"\nLoading tokenizer for {model_checkpoint}...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("Tokenizer loaded.")

# --- Preprocessing Function (ensure available from Module 0) ---
# def preprocess_function(examples): ...
print("\nDefining preprocessing function...") # Added print statement
def preprocess_function(examples):
    inputs = [task_prefix + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs, truncation=True, padding=False, max_length=512)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_structure"], truncation=True, padding=False, max_length=256)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
print("Preprocessing function defined.")


# --- Apply Preprocessing ---
print("\nApplying preprocessing function to V4 datasets...")
tokenized_datasets_v4 = raw_datasets_v4.map(preprocess_function, batched=True,
                                          remove_columns=raw_datasets_v4["train"].column_names)
print("Preprocessing complete for V4 data.")

# --- Load Quantized Base Model ---
print(f"\nLoading Quantized (4-bit) PyTorch base model {model_checkpoint}...")
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_checkpoint,
        quantization_config=bnb_config, # Apply 4-bit config
        device_map="auto" # Recommended for bitsandbytes integration
        # No explicit .to(device) needed when using device_map="auto"
    )
    print(f"Quantized Base model ({model_checkpoint}) loaded.")
except Exception as e:
    print(f"Error loading Quantized PyTorch model {model_checkpoint}: {e}")
    if isinstance(e, torch.cuda.OutOfMemoryError): torch.cuda.empty_cache()
    raise SystemExit("Quantized model loading failed.")


# --- Apply PEFT/LoRA ---
print("\nApplying PEFT/LoRA configuration...")
try:
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )
    # Note: prepare_model_for_kbit_training is often used for CausalLM,
    # less consistently needed for Seq2Seq with device_map="auto" and LoRA.
    # Let's try without it first. If errors occur, we might add it back.
    # model = prepare_model_for_kbit_training(model) # Keep commented out for now
    model = get_peft_model(model, lora_config)
    print("PEFT model created successfully.")
    model.print_trainable_parameters()
except Exception as e:
    print(f"Error applying PEFT: {e}")
    raise SystemExit("PEFT application failed.")

# --- Data Collator ---
# Need to ensure tokenizer is passed if model doesn't have it after PEFT?
# Trainer usually handles this. Let's keep it simple.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
print("\nData collator defined.")

# --- Define Training Arguments ---
print("\nDefining Training Arguments for V4 QLoRA...")
training_args = Seq2SeqTrainingArguments(
    output_dir=qlora_output_dir_v4, # Use QLoRA output dir
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE, # Try 16
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE * 2, # 32
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, # 1
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch", # Keep working strategy name
    save_strategy="epoch",
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=FP16_ENABLED, # Use FP16 compute on T4
    # bf16=False, # Explicitly ensure BF16 is off if FP16 is on for T4
    logging_dir=logging_dir_pt_v4,
    logging_strategy="steps",
    logging_steps=500,
    report_to="tensorboard",
    push_to_hub=False,
    # gradient_checkpointing=True, # Optional: Can add if still memory issues persist
)
print("Training Arguments defined.")

# --- Initialize Trainer ---
print("\nInitializing Trainer with QLoRA PEFT model...")
trainer = Seq2SeqTrainer(
    model=model, # PEFT model
    args=training_args,
    train_dataset=tokenized_datasets_v4["train"], # V4 data
    eval_dataset=tokenized_datasets_v4["validation"], # V4 data
    tokenizer=tokenizer,
    data_collator=data_collator,
)
print("Trainer initialized.")

# --- Train ---
print(f"\nStarting QLoRA model training on V4 data ({EPOCHS} epochs)...")
print(f"Dataset size: ~{len(tokenized_datasets_v4['train']) // 1000}k train, ~{len(tokenized_datasets_v4['validation']) // 1000}k validation.")
print("Training should be more memory efficient. Speed TBD.")
start_train_time = time.time()
training_successful = False
try:
    train_result = trainer.train()
    print("\nTraining complete.")
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    training_successful = True
except Exception as e:
    print(f"An error occurred during training: {e}")
    import traceback; traceback.print_exc()
    print("Training interrupted. Check output directory for checkpoints.")

end_train_time = time.time()
print(f"Total training time: {end_train_time - start_train_time:.2f} seconds")

# --- Save Final (Best) Adapter ---
if training_successful or os.path.exists(os.path.join(qlora_output_dir_v4, 'adapter_model.safetensors')):
    print(f"\nSaving best QLoRA adapter from V4 training run to {final_adapter_dir_v4}...")
    try:
        trainer.save_model(final_adapter_dir_v4)
        tokenizer.save_pretrained(final_adapter_dir_v4)
        print("Best V4 QLoRA adapter and tokenizer saved.")
    except Exception as e: print(f"Error saving final adapter: {e}")
else: print("\nSkipping final adapter saving.")

print(f"\n--- Module 4 (V4 QLoRA {model_checkpoint}): Run Finished ---")

--- Setting up Module 4 (V4 QLoRA): Training t5-base with 4-bit Quantization + LoRA ---
PyTorch version: 2.5.1+cu124

Checking for GPU availability for PyTorch...
GPU Available! Device: Tesla T4
Tesla T4 detected, proceeding with QLoRA setup.

BitsAndBytesConfig for 4-bit quantization defined.

--- Configuration for V4 QLoRA Run (t5-base) ---
Loading V4 data from /kaggle/working/train_data_v4.csv and /kaggle/working/val_data_v4.csv
Using base model: t5-base
Quantization: 4-bit NF4, Compute dtype: float16
PEFT Method: LoRA (r=16, alpha=32, modules=['q', 'v'])
Epochs: 3
Batch Size (Per Device): 16
Effective Batch Size: 16
Learning Rate: 0.0001
FP16 Enabled: True
Adapter Output/Checkpoint Dir: /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA
Final Adapter Save Dir: /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA/final_adapter
---------------------------------

Loaded V4 train data: (154282, 2)
Loaded V4 validation data: (38571, 2)

Converte

Map:   0%|          | 0/154282 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/38571 [00:00<?, ? examples/s]

Preprocessing complete for V4 data.

Loading Quantized (4-bit) PyTorch base model t5-base...
Quantized Base model (t5-base) loaded.

Applying PEFT/LoRA configuration...
PEFT model created successfully.
trainable params: 1,769,472 || all params: 224,673,024 || trainable%: 0.7876

Data collator defined.

Defining Training Arguments for V4 QLoRA...
Training Arguments defined.

Initializing Trainer with QLoRA PEFT model...


/tmp/ipykernel_246/1545138086.py:211: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Trainer initialized.

Starting QLoRA model training on V4 data (3 epochs)...
Dataset size: ~154k train, ~38k validation.
Training should be more memory efficient. Speed TBD.


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,1.584600,1.449326
2,1.542600,1.398750
3,1.517200,1.383616



Training complete.
***** train metrics *****
  epoch                    =        3.0
  total_flos               = 15674516GF
  train_loss               =     1.5969
  train_runtime            = 2:45:59.10
  train_samples_per_second =     46.475
  train_steps_per_second   =      2.905
Total training time: 9959.53 seconds

Saving best QLoRA adapter from V4 training run to /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA/final_adapter...
Best V4 QLoRA adapter and tokenizer saved.

--- Module 4 (V4 QLoRA t5-base): Run Finished ---


In [7]:
# Module 5 (Revised V4): Inference and Testing (QLoRA V4 Model)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import PeftModel
import time
import pprint
import os
import re # Added re for the parser within the function

print("--- Starting Module 5 (Revised V4): Inference (QLoRA Model) ---")

# --- Configuration ---
base_model_name = "t5-base"
# --- Path to V4 QLoRA Adapter ---
adapter_dir = f"/kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-{base_model_name}-QLoRA/final_adapter"
task_prefix = "generate sections: "

# --- Check if adapter files exist ---
print(f"Checking for V4 QLoRA adapter files in: {adapter_dir}")
config_path = os.path.join(adapter_dir, 'adapter_config.json')
# Check for either safetensors or bin weights file
weights_path_st = os.path.join(adapter_dir, 'adapter_model.safetensors')
weights_path_bin = os.path.join(adapter_dir, 'adapter_model.bin')

if not os.path.exists(config_path) or not (os.path.exists(weights_path_st) or os.path.exists(weights_path_bin)):
    print(f"ERROR: V4 QLoRA PEFT adapter files not found in {adapter_dir}.")
    # Simple fallback check - did it save directly to output dir instead of final_adapter?
    adapter_dir_fallback = f"/kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-{base_model_name}-QLoRA"
    config_path_fb = os.path.join(adapter_dir_fallback, 'adapter_config.json')
    if os.path.exists(config_path_fb):
         print(f"Files not in /final_adapter, trying main dir: {adapter_dir_fallback}")
         adapter_dir = adapter_dir_fallback
    else:
        print("Please ensure Module 4 (V4 QLoRA) completed and saved the adapter.")
        raise SystemExit("Failed to find essential PEFT adapter files.")

# --- Determine Device ---
if torch.cuda.is_available():
     device = torch.device("cuda");
     print(f"GPU Available! Using device: {torch.cuda.get_device_name(0)}")
else:
     device = torch.device("cpu");
     print("Warning: No GPU detected. Inference might be slow.")

# --- Load Base Model (Can load in 8-bit for faster/lighter inference if needed) ---
# Decide if we need quantization for inference too. For consistency/speed, let's try 8-bit.
print(f"\nLoading Tokenizer from {adapter_dir}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    print("Tokenizer loaded.")
except Exception as e:
     print(f"Error loading tokenizer from {adapter_dir}: {e}"); raise SystemExit("Tokenizer loading failed.")

print(f"Loading Base Model {base_model_name} (with 8-bit quantization for inference)...")
# Optional: Use 8-bit for inference to save memory/potentially speed up
bnb_config_inf = BitsAndBytesConfig(load_in_8bit=True)
try:
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config_inf, # Load base in 8-bit
        device_map="auto" # Let accelerate handle placement
    )
    print("Base model loaded in 8-bit.")
except Exception as e:
    print(f"Error loading base model with 8-bit quantization: {e}")
    print("Attempting to load base model without quantization...")
    try:
        base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
        base_model.to(device) # Move manually if not using device_map
        print("Base model loaded without quantization.")
    except Exception as e2:
        print(f"Error loading base model without quantization: {e2}")
        raise SystemExit("Base model loading failed.")


# --- Load V4 QLoRA Adapter ---
print(f"Loading V4 QLoRA adapter from {adapter_dir} onto base model...")
try:
    # When loading PEFT adapter onto quantized model, usually device_map needed on base load
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    # If base model wasn't loaded with device_map, ensure final model is on device
    if not hasattr(model, 'hf_device_map'): model.to(device)
    model.eval()
    print(f"V4 QLoRA Model ready (on device: {model.device}).")
except Exception as e:
    print(f"Error loading V4 QLoRA adapter: {e}")
    raise SystemExit("Failed to load adapter.")

# --- Generation Function (including parser) ---
def generate_hierarchical_structure_qlora(article_title, article_description=""):
    print(f"\nInput Title: '{article_title}'")
    if article_description: print(f"Input Desc: '{article_description}'"); input_text = f"{article_title} [SEP] {article_description}".strip()
    else: input_text = article_title

    full_prompt = task_prefix + input_text
    print(f"Model Input Prompt: '{full_prompt}'")
    start_time = time.time()

    inputs = tokenizer(full_prompt, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device) # Use model's device
    input_ids = inputs['input_ids']; attention_mask = inputs['attention_mask']

    raw_output = "Error during generation."; parsed_structure = []
    try:
        with torch.no_grad():
            output_sequences = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=256, # Max length for generated output
                num_beams=4,    # Use beam search
                repetition_penalty=1.7, # Penalize repetition
                early_stopping=True
            )
        raw_output = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
        end_time = time.time()
        print(f"Raw Generated Output: '{raw_output}'")
        print(f"Generation time: {end_time - start_time:.2f} seconds")

        # --- Parsing Logic (same as before) ---
        parts = raw_output.split(' | ')
        for part in parts:
            part = part.strip()
            if part.startswith('(') and part.endswith(')'):
                 content = part[1:-1]
                 if ': ' in content:
                     try: level_str, title = content.split(': ', 1); level = int(level_str); parsed_structure.append((level, title.strip()))
                     except ValueError as parse_err: print(f"Warning: Could not parse level/title from part '{part}': {parse_err}")
                 else: print(f"Warning: Separator ': ' not found in content of '{part}'")
            elif part: print(f"Warning: Output part '{part}' does not match expected format '(level: title)'")
    except Exception as e:
        print(f"Error during generation or parsing: {e}"); raw_output = f"Error: {e}"

    print("Parsed Structure (if successful):"); pprint.pprint(parsed_structure)
    return raw_output, parsed_structure

# --- Test Cases ---
print("\n--- Running Test Cases on V4 QLoRA Model ---")
test_inputs = [
    {"title": "Quantum Computing", "desc": "Area of study focused on the development of computer based on the principles of quantum theory"},
    {"title": "History of Mumbai", "desc": "History of the capital city of the Indian state of Maharashtra"},
    {"title": "Marie Curie", "desc": "Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity"},
    {"title": "Photosynthesis", "desc": "Process used by plants and other organisms to convert light energy into chemical energy"},
    {"title": "1999 Cricket World Cup", "desc": "The seventh edition of the Cricket World Cup, organized by the International Cricket Council (ICC)"},
    {"title": "James Webb Space Telescope", "desc": "Space telescope designed primarily to conduct infrared astronomy"},
    {"title": "Machine Learning", "desc": "Study of algorithms and statistical models that computer systems use to perform tasks without explicit instructions"},
    {"title": "Data Structure", "desc": "A data organization, management, and storage format that enables efficient access and modification"}
]
generated_outputs_v4_qlora = {}
for item in test_inputs:
    title = item["title"]; desc = item.get("desc", "")
    raw_output, parsed_output = generate_hierarchical_structure_qlora(title, desc) # Use updated function name
    generated_outputs_v4_qlora[title] = {'raw': raw_output, 'parsed': parsed_output}

print("\n--- Module 5 (Revised V4 - QLoRA): Inference Complete ---")

--- Starting Module 5 (Revised V4): Inference (QLoRA Model) ---
Checking for V4 QLoRA adapter files in: /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA/final_adapter
GPU Available! Using device: Tesla T4

Loading Tokenizer from /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA/final_adapter...
Tokenizer loaded.
Loading Base Model t5-base (with 8-bit quantization for inference)...
Base model loaded in 8-bit.
Loading V4 QLoRA adapter from /kaggle/working/t5-wiki-structure-generator-pt-v4-filtered-t5-base-QLoRA/final_adapter onto base model...
V4 QLoRA Model ready (on device: cuda:0).

--- Running Test Cases on V4 QLoRA Model ---

Input Title: 'Quantum Computing'
Input Desc: 'Area of study focused on the development of computer based on the principles of quantum theory'
Model Input Prompt: 'generate sections: Quantum Computing [SEP] Area of study focused on the development of computer based on the principles of quantum theory'
Raw Generated 